# Kaggle 2xT4 Full Repro

Notebook ini menjalankan semuanya lewat notebook:
1. Clone repo ke /kaggle/temp
2. Setup .venv dan install requirements.txt
3. Download parent teacher model dari Hugging Face
4. Gabung metadata.csv + metadata_indsp.csv
5. Prepare dataset Arrow
6. Setup W&B hardcoded key
7. Distillation training di 2x T4

Semua command Python dijalankan via: uv run --python .venv/bin/python

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

# ================= User Config =================
REPO_URL = "https://github.com/<username>/kcv-tts.git"
REPO_BRANCH = "main"

WANDB_API_KEY = "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"
WANDB_ENTITY = "haidarmuhammaddzaky-institut-teknologi-sepuluh-nopember"
WANDB_PROJECT = "kaceve"

HF_REPO_ID = "Eempostor/F5-TTS-INDO-FINETUNE-V2"
HF_CKPT_FILENAME = "f5_tts_indo_v2.pt"

CSV_1 = Path("/kaggle/input/datasets/benedictusryugunawan/tts-indo/data/metadata.csv")
CSV_2 = Path("/kaggle/input/datasets/benedictusryugunawan/tts-indo/data/metadata_indsp.csv")

WORKDIR = Path("/kaggle/temp")
REPO_DIR = WORKDIR / "kcv-tts"
VENV_DIR = REPO_DIR / ".venv"
VENV_PY = VENV_DIR / "bin/python"

HF_OUT_DIR = REPO_DIR / "ckpts/hf/Eempostor_F5-TTS-INDO-FINETUNE-V2"
TEACHER_CKPT = HF_OUT_DIR / HF_CKPT_FILENAME
MERGED_CSV = REPO_DIR / "data/metadata_merged.csv"
PREPARED_DATASET_DIR = REPO_DIR / "data/datasetku_pinyin"

def run_cmd(cmd, cwd=None, env=None):
    printable = cmd if isinstance(cmd, str) else " ".join(str(x) for x in cmd)
    print("\n$", printable)
    return subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        env=env,
        check=True,
        text=True,
    )

def run_py(args, cwd=None, env=None):
    return run_cmd(["uv", "run", "--python", str(VENV_PY), *args], cwd=cwd, env=env)

print("Config siap.")
print("CSV_1 exists:", CSV_1.exists())
print("CSV_2 exists:", CSV_2.exists())

In [ ]:
# 1) Clone repo ke /kaggle/temp
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

run_cmd(["git", "clone", "--recursive", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)])
run_cmd(["ls", "-lah", str(REPO_DIR)])

In [ ]:
# 2) Setup .venv dan install requirements.txt hasil freeze
if shutil.which("uv") is None:
    run_cmd(["python3", "-m", "pip", "install", "-U", "uv"])

run_cmd(["uv", "venv", str(VENV_DIR)])
run_cmd(["uv", "pip", "install", "--python", str(VENV_PY), "--upgrade", "pip", "setuptools", "wheel"])

req_in = REPO_DIR / "requirements.txt"
req_out = REPO_DIR / "requirements.kaggle.txt"

with req_in.open("r", encoding="utf-8") as f:
    lines = f.readlines()

filtered = []
for line in lines:
    stripped = line.strip()
    if not stripped or stripped.startswith("#"):
        filtered.append(line)
        continue
    # Drop local editable refs produced by freeze on dev machine.
    if stripped.startswith("-e ") and "file://" in stripped:
        continue
    if " @ file://" in stripped:
        continue
    filtered.append(line)

req_out.write_text("".join(filtered), encoding="utf-8")
print("Using requirements file:", req_out)

run_cmd(["uv", "pip", "install", "--python", str(VENV_PY), "-r", str(req_out)])

In [ ]:
# 3) Validasi GPU di Kaggle
run_py([
    "-c",
    "import torch; print('torch', torch.__version__); print('cuda_count', torch.cuda.device_count()); [print(i, torch.cuda.get_device_name(i)) for i in range(torch.cuda.device_count())]",
])

In [ ]:
# 4) Download parent teacher model dari HF
run_cmd(["uv", "pip", "install", "--python", str(VENV_PY), "-U", "huggingface_hub"])

download_script = "\n".join([
    "from pathlib import Path",
    "from huggingface_hub import hf_hub_download",
    f"repo_id = {HF_REPO_ID!r}",
    f"filename = {HF_CKPT_FILENAME!r}",
    f"out_dir = Path(r'{HF_OUT_DIR}')",
    "out_dir.mkdir(parents=True, exist_ok=True)",
    "path = hf_hub_download(repo_id=repo_id, filename=filename, local_dir=str(out_dir), local_dir_use_symlinks=False)",
    "print(path)",
])

run_py(["-c", download_script], cwd=REPO_DIR)
run_cmd(["ls", "-lah", str(HF_OUT_DIR)])

In [ ]:
# 5) Merge 2 CSV metadata -> audio_file|text (pipe-delimited)
import pandas as pd

def load_metadata(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, sep=None, engine="python")
    df.columns = [c.strip() for c in df.columns]

    audio_candidates = ["audio_file", "audio_path", "wav_path", "path", "file"]
    text_candidates = ["text", "transcript", "sentence", "normalized_text", "utterance"]

    audio_col = next((c for c in audio_candidates if c in df.columns), None)
    text_col = next((c for c in text_candidates if c in df.columns), None)

    if audio_col is None or text_col is None:
        raise ValueError(f"Kolom tidak cocok di {path}. Dapat: {list(df.columns)}")

    out = df[[audio_col, text_col]].copy()
    out.columns = ["audio_file", "text"]
    out["audio_file"] = out["audio_file"].astype(str).str.strip()
    out["text"] = out["text"].astype(str).str.strip()
    out = out[(out["audio_file"] != "") & (out["text"] != "")]

    source_dir = path.parent
    def absolutize(p: str) -> str:
        pp = Path(p).expanduser()
        if pp.is_absolute():
            return str(pp)
        return str((source_dir / pp).resolve())

    out["audio_file"] = out["audio_file"].map(absolutize)
    out["source_csv"] = str(path)
    return out

df1 = load_metadata(CSV_1)
df2 = load_metadata(CSV_2)
merged = pd.concat([df1, df2], ignore_index=True).drop_duplicates(subset=["audio_file", "text"])

exists_mask = merged["audio_file"].map(lambda p: Path(p).exists())
missing = int((~exists_mask).sum())
if missing:
    print(f"Dropping {missing} rows with missing audio paths.")
merged = merged[exists_mask].copy()

MERGED_CSV.parent.mkdir(parents=True, exist_ok=True)
merged[["audio_file", "text"]].to_csv(MERGED_CSV, sep="|", index=False)

print("Merged rows (after drop missing):", len(merged))
print("Saved:", MERGED_CSV)
print("Source CSVs:", CSV_1, CSV_2)
print(merged[["audio_file", "text"]].head(5))

In [ ]:
# 6) Jalankan prepare_csv_wavs.py di notebook
PREPARED_DATASET_DIR.mkdir(parents=True, exist_ok=True)
run_py([
    "src/f5_tts/train/datasets/prepare_csv_wavs.py",
    str(MERGED_CSV),
    str(PREPARED_DATASET_DIR),
    "--workers",
    "8",
], cwd=REPO_DIR)

run_cmd(["ls", "-lah", str(PREPARED_DATASET_DIR)])

In [ ]:
# 7) W&B hardcoded login + sanity logging
env = os.environ.copy()
env["WANDB_API_KEY"] = WANDB_API_KEY
env["WANDB_ENTITY"] = WANDB_ENTITY

wandb_smoke = f"""
import random
import wandb

wandb.login(key={WANDB_API_KEY!r})
run = wandb.init(
    entity={WANDB_ENTITY!r},
    project={WANDB_PROJECT!r},
    config={{
        "learning_rate": 0.02,
        "architecture": "CNN",
        "dataset": "CIFAR-100",
        "epochs": 10,
    }},
)
epochs = 10
offset = random.random() / 5
for epoch in range(2, epochs):
    acc = 1 - 2**-epoch - random.random() / epoch - offset
    loss = 2**-epoch + random.random() / epoch + offset
    run.log({"acc": acc, "loss": loss})
run.finish()
print("wandb sanity done")
""".strip()

run_py(["-c", wandb_smoke], cwd=REPO_DIR, env=env)

In [ ]:
# 8) Distill phase (final checkpoint only), lalu full training phase (final checkpoint only)
env = os.environ.copy()
env["WANDB_API_KEY"] = WANDB_API_KEY
env["WANDB_ENTITY"] = WANDB_ENTITY

CKPT_ROOT_WORKING = Path("/kaggle/working/ckpts")
DISTILL_TAG = "distill_final_datasetku"
FULL_TAG = "full_final_datasetku"

distill_dir_abs = CKPT_ROOT_WORKING / DISTILL_TAG
full_dir_abs = CKPT_ROOT_WORKING / FULL_TAG

# train.py menyimpan checkpoint ke repo_root/ckpts.save_dir, jadi kita pakai relative escape ke /kaggle/working
distill_save_dir_rel = f"../../working/ckpts/{DISTILL_TAG}"
full_save_dir_rel = f"../../working/ckpts/{FULL_TAG}"

for out_dir in [distill_dir_abs, full_dir_abs]:
    out_dir.mkdir(parents=True, exist_ok=True)
    for old in out_dir.glob("model_*.pt"):
        old.unlink()
    last_ckpt = out_dir / "model_last.pt"
    if last_ckpt.exists():
        last_ckpt.unlink()
    for old_pretrained in out_dir.glob("pretrained_*.pt"):
        old_pretrained.unlink()
    for old_pretrained_sf in out_dir.glob("pretrained_*.safetensors"):
        old_pretrained_sf.unlink()

no_periodic_ckpt_overrides = [
    "ckpts.save_per_updates=999999999",
    "ckpts.last_per_updates=999999999",
    "ckpts.keep_last_n_checkpoints=0",
    "ckpts.log_samples=False",
]

# ---- Phase A: Distillation ----
distill_cmd = [
    "uv",
    "run",
    "--python",
    str(VENV_PY),
    "accelerate",
    "launch",
    "--num_processes=2",
    "--mixed_precision=fp16",
    "src/f5_tts/train/train.py",
    "--config-name",
    "F5TTS_HYBRID_DISTILL_5K.yaml",
    "datasets.name=datasetku",
    f"model.cfm_experiment.teacher_ckpt_path={TEACHER_CKPT}",
    "datasets.batch_size_per_gpu=768",
    "datasets.max_samples=1",
    "datasets.num_workers=4",
    "optim.epochs=1",
    "optim.num_warmup_updates=500",
    "ckpts.logger=wandb",
    f"ckpts.wandb_project={WANDB_PROJECT}",
    "ckpts.wandb_run_name=F5TTS_HYBRID_DISTILL_5K_datasetku_kaggle_2xT4",
    f"ckpts.save_dir={distill_save_dir_rel}",
    *no_periodic_ckpt_overrides,
]

run_cmd(distill_cmd, cwd=REPO_DIR, env=env)
run_cmd(["ls", "-lah", str(distill_dir_abs)])

distill_last = distill_dir_abs / "model_last.pt"
if not distill_last.exists():
    raise FileNotFoundError(f"Distill final checkpoint tidak ditemukan: {distill_last}")

# full training BUKAN dari nol: partial init dari parent checkpoint yang sama dengan teacher distill
if not TEACHER_CKPT.exists():
    raise FileNotFoundError(f"Parent checkpoint tidak ditemukan: {TEACHER_CKPT}")

pretrained_for_full = full_dir_abs / "pretrained_indo_v2.pt"
shutil.copy2(TEACHER_CKPT, pretrained_for_full)
print("Distill final checkpoint:", distill_last)
print("Prepared full-training partial init checkpoint (same parent as distill):", pretrained_for_full)

# ---- Phase B: Full training ----
full_cmd = [
    "uv",
    "run",
    "--python",
    str(VENV_PY),
    "accelerate",
    "launch",
    "--num_processes=2",
    "--mixed_precision=fp16",
    "src/f5_tts/train/train.py",
    "--config-name",
    "F5TTS_EarlyBiMamba_v1.yaml",
    "datasets.name=datasetku",
    "datasets.batch_size_per_gpu=768",
    "datasets.max_samples=1",
    "datasets.num_workers=4",
    "optim.epochs=1",
    "optim.num_warmup_updates=500",
    "ckpts.logger=wandb",
    f"ckpts.wandb_project={WANDB_PROJECT}",
    "ckpts.wandb_run_name=F5TTS_EarlyBiMamba_v1_datasetku_kaggle_2xT4",
    f"ckpts.save_dir={full_save_dir_rel}",
    *no_periodic_ckpt_overrides,
]

run_cmd(full_cmd, cwd=REPO_DIR, env=env)
run_cmd(["ls", "-lah", str(full_dir_abs)])

full_last = full_dir_abs / "model_last.pt"
if not full_last.exists():
    raise FileNotFoundError(f"Full-training final checkpoint tidak ditemukan: {full_last}")

print("\nFinal checkpoints:")
print("- Distill final:", distill_last)
print("- Full final:", full_last)

## Notes
- Kalau OOM, turunkan datasets.batch_size_per_gpu ke 640 atau 512.
- Checkpoint hanya dibuat saat akhir fase distill dan akhir fase full training (tanpa checkpoint periodik).
- Full training tidak dari nol: inisialisasi partial load dari parent checkpoint yang sama dengan teacher distill (Eempostor/F5-TTS-INDO-FINETUNE-V2).
- Lokasi final checkpoint: /kaggle/working/ckpts/distill_final_datasetku/model_last.pt dan /kaggle/working/ckpts/full_final_datasetku/model_last.pt.
- Proses kerja lain tetap berjalan dari /kaggle/temp.